# FinPilot ? analyse exploratoire

Le d?p?t ne fournit aucun client, aucune transaction et aucun montant fictif. Pour ex?cuter ce notebook, d?marrez FinPilot localement puis saisissez ou importez des donn?es dans votre base SQLite. Le fichier `backend/finpilot.db` est ignor? par Git et ne doit pas ?tre ajout? au d?p?t public.


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path('../backend/finpilot.db')
con = sqlite3.connect(db_path)
tx = pd.read_sql_query('SELECT kind, amount_xof, transaction_date, status, activity_id, client_id FROM transactions', con, parse_dates=['transaction_date'])
activities = pd.read_sql_query('SELECT id, name FROM activities', con)
tx = tx.merge(activities, left_on='activity_id', right_on='id', how='left')
tx.head()

## Flux mensuels
On compare les encaissements reçus aux dépenses payées. Les lignes en attente sont exclues afin de ne pas confondre prévision et trésorerie réalisée.

In [ ]:
settled = tx[tx.status == 'received'].copy()
settled['month'] = settled.transaction_date.dt.to_period('M').astype(str)
monthly = settled.pivot_table(index='month', columns='kind', values='amount_xof', aggfunc='sum', fill_value=0)
monthly['net_cash'] = monthly.get('income', 0) - monthly.get('expense', 0)
monthly.tail(18).plot(kind='bar', figsize=(14, 5), title='Encaissements et dépenses mensuels')

## Contribution par activité
Cette vue identifie les activités qui contribuent le plus au résultat net de caisse. L'affectation des dépenses reste un préalable à une comparaison complète.

In [ ]:
by_activity = settled.pivot_table(index='name', columns='kind', values='amount_xof', aggfunc='sum', fill_value=0)
by_activity['net_cash'] = by_activity.get('income', 0) - by_activity.get('expense', 0)
by_activity.sort_values('net_cash', ascending=False)